# 🎙️ Voice ChatGPT — Whisper + gTTS + OpenAI
**Desafio de Projeto DIO**  
**Autora:** Thayná Batista da Silva  
**Curso:** Análise e Desenvolvimento de Sistemas — Faculdade Senac Recife-PE (2025–2027)  

---
Este notebook integra:
- 🧠 **Whisper** — Speech-to-Text (transcrição de voz)
- 💬 **ChatGPT** — Processamento de linguagem natural
- 🔊 **gTTS** — Text-to-Speech (síntese de voz)


In [ ]:
# ─── 1. Instalação de dependências ───────────────────────────────────────────
!pip install openai gtts playsound SpeechRecognition -q

In [ ]:
# ─── 2. Imports e configuração ───────────────────────────────────────────────
import os
import openai
from gtts import gTTS
from IPython.display import Audio, display
import tempfile

# Insira sua chave de API aqui ou use secrets do Colab
openai.api_key = "SUA_OPENAI_API_KEY_AQUI"

IDIOMA_RESPOSTA = "pt"
MODELO_GPT      = "gpt-3.5-turbo"
MODELO_WHISPER  = "whisper-1"
print('✅ Configuração concluída!')

In [ ]:
# ─── 3. Gravar áudio no Colab (via JavaScript) ───────────────────────────────
from IPython.display import Javascript
from google.colab import output
import base64

RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
const b64  = blob => new Promise(res => {
  const r = new FileReader(); r.onload = () => res(r.result); r.readAsDataURL(blob);
});
const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
const rec    = new MediaRecorder(stream, { mimeType: 'audio/webm' });
const chunks = [];
rec.ondataavailable = e => chunks.push(e.data);
rec.start();
console.log('🎤 Gravando por 5 segundos...');
await sleep(5000);
rec.stop();
await sleep(500);
const blob   = new Blob(chunks);
const data64 = await b64(blob);
window._audioData = data64;
google.colab.kernel.invokeFunction('notebook.get_audio', [data64], {});
"""

audio_data_global = {}

def get_audio(data):
    audio_data_global['data'] = data

output.register_callback('notebook.get_audio', get_audio)

def gravar_audio_colab():
    display(Javascript(RECORD_JS))
    import time; time.sleep(7)
    return audio_data_global.get('data', '')

print('✅ Função de gravação pronta!')

In [ ]:
# ─── 4. Transcrição com Whisper ──────────────────────────────────────────────
def transcrever_audio(audio_b64: str) -> str:
    header, data = audio_b64.split(',', 1)
    audio_bytes  = base64.b64decode(data)
    with tempfile.NamedTemporaryFile(suffix='.webm', delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name
    try:
        with open(tmp_path, 'rb') as f:
            resp = openai.Audio.transcribe(model=MODELO_WHISPER, file=f)
        texto = resp['text'].strip()
        print(f'📝 Você disse: {texto}')
        return texto
    finally:
        os.remove(tmp_path)

print('✅ Função de transcrição pronta!')

In [ ]:
# ─── 5. Resposta via ChatGPT ─────────────────────────────────────────────────
historico = []

def perguntar_ao_chatgpt(pergunta: str) -> str:
    historico.append({'role': 'user', 'content': pergunta})
    resp = openai.ChatCompletion.create(
        model=MODELO_GPT,
        messages=[
            {'role': 'system', 'content': 'Você é um assistente amigável. Responda em português de forma clara e concisa.'},
            *historico,
        ]
    )
    texto = resp['choices'][0]['message']['content'].strip()
    historico.append({'role': 'assistant', 'content': texto})
    print(f'🤖 ChatGPT: {texto}')
    return texto

print('✅ Função ChatGPT pronta!')

In [ ]:
# ─── 6. Síntese de voz com gTTS ──────────────────────────────────────────────
def falar_resposta(texto: str):
    tts = gTTS(text=texto, lang=IDIOMA_RESPOSTA, slow=False)
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as tmp:
        tts.save(tmp.name)
        tmp_path = tmp.name
    display(Audio(tmp_path, autoplay=True))
    os.remove(tmp_path)

print('✅ Função gTTS pronta!')

In [ ]:
# ─── 7. EXECUTAR — Conversa por Voz ─────────────────────────────────────────
print('=' * 50)
print('  🎙️  Voice ChatGPT — Execute esta célula para falar!')
print('=' * 50)

audio_b64 = gravar_audio_colab()
if audio_b64:
    pergunta  = transcrever_audio(audio_b64)
    resposta  = perguntar_ao_chatgpt(pergunta)
    falar_resposta(resposta)
else:
    print('⚠️ Nenhum áudio capturado. Tente novamente.')